# Quark-Gluons jets

This notebook is dedicated to exploring the dataset and visualizing the features to gain insights into the data distribution and relationships between variables. We will use various plotting techniques to understand the characteristics of the dataset and identify any patterns or anomalies that may be present. 

The 3 main datasets wil be:
- **Top tagging**: This dataset contains information about top quark tagging, which is a technique used in particle physics to identify top quarks in high-energy collisions. The features in this dataset may include various kinematic variables and jet substructure observables that are relevant for top quark identification.
- **Quark-gluon jet tagging**: This dataset contains information about quark-gluon jet tagging, which is a technique used to distinguish between jets originating from quarks and gluons. The features in this dataset may include various jet substructure observables and kinematic variables that are relevant for quark-gluon discrimination.
- **Higgs tagging**: This dataset contains information about Higgs boson tagging, which is a technique used to identify Higgs bosons in high-energy collisions. The features in this dataset may include various kinematic variables and jet substructure observables that are relevant for Higgs boson identification.

We can find the dataset in the following path: `../data/raw/quark-gluon/QG_jets_fp32_0.npz`. We use only 1 file to this exploration.

**Description taken from the original source:** [zenodo](https://zenodo.org/records/19362155) 

A `float32` (single-precision) version of the quark and gluon jet dataset originally published by Komiske, Metodiev, and Thaler (Zenodo record 3164691). Only the 20-file subset without charm and bottom quark jets is included here. All simulation parameters and jet selection criteria are identical to the original:

- Pythia 8.226, $\sqrt{s} = 14 TeV$
- Quarks from `WeakBosonAndParton:qg2gmZq`, gluons from `WeakBosonAndParton:qqbar2gmZg` with the Z decaying to neutrinos
- FastJet 3.3.0, anti-k_t jets with $R = 0.4$
- $p_T^jet \in [500, 550] GeV, |y^jet| < 1.7$

There are 20 files, each in compressed NumPy format (`QG_jets_fp32_0.npz` through `QG_jets_fp32_19.npz`). Each file contains two arrays:

- **X**: (100000, M, 4) — 50k quark and 50k gluon jets, randomly sorted, padded to max multiplicity M, with particle features (pt, rapidity, azimuthal angle, pdgid) stored as float32
- **y**: (100000,) — jet labels, gluon = 0, quark = 1
The original dataset stores X in float64. Here X has been cast to float32, approximately halving file size. The y labels are unchanged.

Komiske, Metodiev, Thaler, Energy Flow Networks: Deep Sets for Particle Jets, JHEP 01 (2019) 121, arXiv:1810.05165

In [ ]:
# import libraries
import numpy as np

In [ ]:
try:
    with np.load('../data/raw/quark-gluon/QG_jets_fp32_0.npz', 'r') as f:
        print("File keys:", list(f.keys()), "\n")

        for key in f.keys():
            print(f"Shape of '{key}': {f[key].shape}")

        X = f['X'][:]
        y = f['y'][:]
except Exception as e:
    print(f"Error loading data using numpy: {e}")

Let's see what is inside

### Data exploration

In [ ]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import gc

In [ ]:
num_jets, max_multiplicity, num_features = X.shape

# Create jet IDs and labels for each particle
jet_ids = np.repeat(np.arange(num_jets), max_multiplicity)
labels = np.repeat(y, max_multiplicity)
# Flatten the 3D array to 2D for easier DataFrame creation
x_flat = X.reshape(-1, num_features)

# dataframe
df = pl.DataFrame(
    x_flat,
    schema=['pt', 'eta', 'phi', 'pdgid']
    ).with_columns([
    pl.Series('jet_id', jet_ids, dtype=pl.Int32),
    pl.Series('target', labels, dtype=pl.Int8)
])

df_reals = df.filter(pl.col('pt') > 0)

del num_features, num_jets, x_flat, labels, jet_ids, df#, max_multiplicity
gc.collect()

print(f"Total particles: {len(df_reals)}")

After this first exploration we found a dataset with $100\ 000 \times 139 = 13\ 900\ 000$ particles in total, but only 4 330 905 are non-zero. This means that the dataset is very sparse, with many particles having zero values for their features, this is likely due to the padding of the jets to a maximum multiplicity.

In [ ]:
# statistics by label
stats = df_reals.group_by('target').agg([
    pl.len().alias('count'),
    pl.col('pt').mean().alias('mean_pt'),
    pl.col('pt').std().alias('std_pt'),
    pl.col('eta').mean().alias('mean_eta'),
    pl.col('eta').std().alias('std_eta'),
    pl.col('phi').mean().alias('mean_phi'),
    pl.col('phi').std().alias('std_phi')
])

print(stats)

We can see some first diferences between gluons, 0; and quarks, 1. For example, the mean transversal momentum for quarks is greater and also the std. From the theory, we expect an isotropic distribution, thatis why mean eta is close to 0. Something interesting is that the mean phi is not close to 0, actually it is close to $\pi$, which is a bit strange. We will explore this later.

## Data visualization

In [ ]:
# style configuration
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 18,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.linewidth": 1.2,
    "xtick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.major.size": 6,
    "ytick.minor.size": 3,
    "ytick.direction": "in",
    "xtick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
    "figure.dpi": 150,
    "figure.figsize": (10, 6),
    "text.usetex": False
})

Q_COLOR = "#1f77b4"
G_COLOR = "#ff7f0e"

### Global features

Now we will visualize the multiplicity

In [ ]:
from matplotlib.gridspec import GridSpec
import pandas as pd
import scipy.stats as stats

In [ ]:
# ==============================================================================
# 1. Data extraction and descriptive statistics with Polars
# ==============================================================================
# Calculate the real multiplicity per jet (with pt > 0)
df_mult = df_reals.group_by(["jet_id", "target"]).agg(
    pl.len().alias("multiplicity")
)

# Separate into NumPy arrays by class
mult_q = df_mult.filter(pl.col("target") == 1)["multiplicity"].to_numpy()
mult_g = df_mult.filter(pl.col("target") == 0)["multiplicity"].to_numpy()

# Extract key metrics for text annotation and markers
min_q, max_q = mult_q.min(), mult_q.max()
min_g, max_g = mult_g.min(), mult_g.max()
mean_q, median_q = mult_q.mean(), np.median(mult_q)
mean_g, median_g = mult_g.mean(), np.median(mult_g)

# ==============================================================================
# 2. HYBRID CANVAS CONFIGURATION (GridSpec)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

# Definition of proportions: 3 parts for histogram, 1 for boxplot
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.00)

ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# Consistent color palette
#Q_COLOR = "#1f77b4"  # Blue (Signal - Quark)
#G_COLOR = "#e377c2"  # Pink/Mint (Background - Gluon)

# ==============================================================================
# 3. UPPER PANEL: STEP HISTOGRAMS
# ==============================================================================
bins_mult = np.arange(0, max_multiplicity + 2, 2)

ax_hist.hist(
    mult_q,
    bins=bins_mult,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=f"Quark Jets ($\mu={mean_q:.1f}$)",
)
ax_hist.hist(
    mult_g,
    bins=bins_mult,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=f"Gluon Jets ($\mu={mean_g:.1f}$)",
)

# Lines for the means
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)

# Upper panel formatting
ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    "Jet Substructure: Multiplicity Dispersions & Padding Analysis",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 4. LOWER PANEL: HYBRID HORIZONTAL BOXPLOTS AND VIOLINPLOTS
# ==============================================================================
# Prepare DataFrame for Seaborn
df_box = pd.DataFrame(
    {
        "Multiplicity": np.concatenate([mult_q, mult_g]),
        "Class": ["Quark (Signal)"] * len(mult_q) + ["Gluon (Bkg)"] * len(mult_g),
    }
)

# Violin layer (Smoothed background density)
sns.violinplot(
    x="Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    bw_method="silverman",
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Narrow Boxplot layer (Exact quartiles)
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}
flierprops = dict(marker=".", markersize=1.5, alpha=0.05, markeredgecolor="gray")

sns.boxplot(
    x="Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    flierprops=flierprops,
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)


# Formato del panel inferior
ax_box.set_xlabel("Jet Particle Multiplicity (Constituent Count)", fontsize=13)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(0, 100)  # Enfocado para ver la física, omitiendo outliers extremos aislados
ax_box.grid(True, linestyle=":", alpha=0.5)

# --- METADATOS ESTILO HEP ---
fig.suptitle(
    r"Pythia 8.226 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Jet Selection Criteria: $p_T^{jet} \in [500, 550]$ GeV, $|y^{jet}| < 1.7$",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. MEMORY CLEANUP
# ==============================================================================
del (df_mult,
    df_box,
    mult_q,
    mult_g,
    mean_q,
    mean_g,
    median_q,
    median_g,
    min_q,
    max_q,
    min_g,
    max_g,
    ax_hist,
    ax_box,
    fig,
    gs,
    meanprops,
    flierprops,
    bins_mult
    )
gc.collect()

In [ ]:
df_mult = df_reals.group_by(['jet_id', "target"]).agg(pl.len().alias("multiplicity"))

mult_g = df_mult.filter(pl.col('target') == 0)['multiplicity'].to_numpy()
mult_q = df_mult.filter(pl.col('target') == 1)['multiplicity'].to_numpy()

# min and max values for the multiplicity distributions
min_mult_g = mult_g.min()
max_mult_g = mult_g.max()
min_mult_q = mult_q.min()
max_mult_q = mult_q.max()

print(f"Gluon Multiplicity: min={min_mult_g}, max={max_mult_g}")
print(f"Quark Multiplicity: min={min_mult_q}, max={max_mult_q}")

del (
    min_mult_g,
    max_mult_g,
    min_mult_q,
    max_mult_q,
    mult_g,
    mult_q,
    df_mult
)
gc.collect()

In this plot we can see a small tail to the right in the Quark distribution, also with a lower dispersion than the Gluon distribution. This is consistent with the fact that Quark jets are more collimated than Gluon jets, which means that they have a smaller angular spread and therefore a lower multiplicity.

The range are pretty similar in both cases.

To continue we will apply a statistical test to see if the difference is significant or not. We will use the Kolmogorov-Smirnov test, which is a non-parametric test that can be used to compare the distributions of two samples. The Kolmogorov-Smirnov test compares the empirical cumulative distribution functions of the two samples, while the Wasserstein distance measures the distance between the two distributions in terms of the amount of "work" required to transform one distribution into the other.

-$H_0$: The two distributions are identical.
-$H_1$: The two distributions are different.

the p-value expected is very low, this is the reason to use a sigma threshold to reject the null hypothesis. We will use a threshold of 5 sigma, which corresponds to a p-value of $3.5 \times 10^{-7}$. This means that if the p-value is less than this threshold, we will reject the null hypothesis and conclude that the two distributions are different.

Other two metrics that we will use to compare the distributions are the Wasserstein distance and the skewness of the Gluon distribution. The Wasserstein distance is a measure of the distance between two probability distributions, while the skewness is a measure of the asymmetry of a distribution. We will also calculate the p-value associated with the skewness test, which tests the null hypothesis that the distribution is symmetric. If the p-value is less than 0.05, we will reject the null hypothesis and conclude that the distribution is not symmetric.

In [ ]:
df_mult = df_reals.group_by(['jet_id', "target"]).agg(pl.len().alias("multiplicity"))

mult_g = df_mult.filter(pl.col('target') == 0)['multiplicity'].to_numpy()
mult_q = df_mult.filter(pl.col('target') == 1)['multiplicity'].to_numpy()

# ==============================================================================
# Statistical Analysis
# ==============================================================================
# 1. Kolmogorov-Smirnov Test (Sensitive to differences in shape and width)
ks_stat, p_val_ks = stats.ks_2samp(mult_g, mult_q)

# Convert p-value to Sigmas (Z-score) safely against memory underflows
if p_val_ks == 0.0:
    # If the p-value is smaller than the 64-bit float limit (2.2e-308)
    sigmas_ks = ">>> 50 sigma (Truncated due to numerical precision)"
else:
    sigmas_ks = f"{stats.norm.ppf(1 - p_val_ks / 2):.2f} sigma"

# 2. Wasserstein Distance (Earth Mover's Distance)
# Quantifies the "work" needed to transform the shape of the gluon into that of the quark
wasserstein_dist = stats.wasserstein_distance(mult_g, mult_q)

# 3. Skewness Test for Gluon Calibration    
skew_gluon = stats.skew(mult_g)
p_val_skew_g = stats.skewtest(mult_g).pvalue

# ==============================================================================
# FORMAL REPORT OF RESULTS (HEP Publication Style)
# ==============================================================================
print("=" * 80)
print(
    f"STATISTICAL SIGNIFICANCE ANALYSIS: JET Multiplicity"
)
print("=" * 80)
print(f"[-] Kolmogorov-Smirnov Statistic D: {ks_stat:.4f}")
print(f"[-] p-value associated with KS test: {p_val_ks}")
print(f"[-] STATISTICAL SIGNIFICANCE (Z)      : {sigmas_ks}")
print("-" * 80)
print(
    f"[-] Wasserstein Distance (EMD)     : {wasserstein_dist:.4f}"
)
print("-" * 80)
print(f"[-] Gluon Symmetry Calibration    :")
print(f"    * Empirical Skewness: {skew_gluon:.4f}")
print(f"    * p-value for pure symmetry (H0): {p_val_skew_g:.4f}")
print("=" * 80)

del (mult_g, mult_q, ks_stat, p_val_ks, sigmas_ks, wasserstein_dist, skew_gluon, p_val_skew_g, df_mult)
gc.collect()

**K-S test results:**

We can see that the p-value is very low, which means that we can reject the null hypothesis and conclude that the two distributions are different. Nevertheless, this diference usign a $p_{value} = 5 \sigma$, could be result from the very large sample size, which is 100k. This is why we will use the Wasserstein distance and the skewness of the Gluon distribution to compare the distributions. The result of $D = 0.5361$ means that the maximum difference between the two empirical cumulative distribution functions is 53.6 %. The Kolmogorov-Smirnov statistic D is defined as:

\begin{equation}
D = \sup_x |F_1(x) - F_2(x)|
\end{equation}

where $F_1(x)$ and $F_2(x)$ are the empirical cumulative distribution functions of the two samples. The value of D ranges from 0 to 1, where 0 indicates that the two distributions are identical and 1 indicates that the two distributions are completely different. The domain of the statistic is the range of the data, which in this case is the range of the multiplicity values, $D: \mathcal{F}_1 \times \mathcal{F}_2 \rightarrow [0, 1]$.

**Wasserstein distance**

The definition of the Wasserstein distance, $W_1: \mathcal{P}_1 \times \mathcal{P}_2 \rightarrow [0, \infty)$, is given by:

\begin{equation}
W(P, Q) = \inf_{\gamma \in \Gamma(P, Q)} \int_{X \times Y} d(x, y) d\gamma(x, y)
\end{equation}

where $P$ and $Q$ are the two probability distributions, $\Gamma(P, Q)$ is the set of all joint distributions $\gamma$ with marginals $P$ and $Q$, and $d(x, y)$ is a distance metric between points $x$ and $y$. The Wasserstein distance measures the minimum amount of "work" required to transform one distribution into the other, where "work" is defined as the product of the distance between points and the amount of probability mass that needs to be moved.

The result of $W = 19.8181$ means that the minimum amount of "work" required to transform the Gluon distribution into the Quark distribution is 19.8181. This is a measure of the difference between the two distributions, where a larger Wasserstein distance indicates a greater difference between the distributions.

**Skewness**

The Skewness of the Gluon distribution, $S:  \mathcal{P} \rightarrow \mathbb{R}$, where $\mathcal{P}$ is the set of all probability distributions with finite third moment, is given by:

\begin{equation}
S = \frac{E[(X - \mu)^3]}{\sigma^3}
\end{equation}

where $E$ is the expected value, $X$ is the random variable, $\mu$ is the mean of the distribution, and $\sigma$ is the standard deviation of the distribution. The skewness measures the asymmetry of a distribution, where a skewness of 0 indicates a symmetric distribution, a positive skewness indicates a distribution with a longer right tail, and a negative skewness indicates a distribution with a longer left tail.

The result of $S = 0.5427$ with $p_{value} \leq 0.0001$ indicates that the skewness is not symmetric, which means that the Gluon distribution has a longer right tail.

Now let's see which particles are contributing to this asymmetry. We will plot the distribution of the particles in the Gluon jets and Quark jets to see if we can identify any patterns or anomalies that may be present.

In [ ]:
df_counts = (
    df_reals.group_by("pdgid")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

# 2. Imprimimos el resultado ordenado
for row in df_counts.iter_rows(named=True):
    print(f"PDG ID: {row['pdgid']}, Count: {row['count']}")
del df_counts
gc.collect()

In [ ]:
# ==============================================================================
# 1. Preprocessing Vectorized with Polars
# ==============================================================================
num_jets, max_multiplicity, _ = X.shape

# Repeat the jet label 'y' for each of its constituents
labels_flat = np.repeat(y, max_multiplicity)
pdgid_flat = X[:, :, 3].reshape(-1)

# Create the initial DataFrame and remove zero padding
df_pdg = pl.DataFrame(
    {"pdgid": pdgid_flat, "target": labels_flat, "pt": X[:, :, 0].reshape(-1)}
).filter(pl.col("pt") > 0)

# Official Particle Data Group (PDG) dictionary for LaTeX rendering
pdg_labels = {
    211: r"$\pi^+$",
    -211: r"$\pi^-$",
    111: r"$\pi^0$",
    22: r"$\gamma$",
    321: r"$K^+$",
    -321: r"$K^-$",
    2212: r"$p$",
    -2212: r"$\bar{p}$",
    11: r"$e^-$",
    -11: r"$e^+$",
    13: r"$\mu^-$",
    -13: r"$\mu^+$",
    2112: r"$n$",
    -2112: r"$\bar{n}$",
    130: r"$K_L^0$",
    310: r"$K_S^0$",
}

# Find the top 8 most abundant particles dynamically
top_particles = (
    df_pdg.group_by("pdgid")
    .len()
    .sort("len", descending=True)
    .head(14)["pdgid"]
    .to_list()
)

# Filter by the top 8 and calculate the fraction normalized by Jet type (target)
df_counts = (
    df_pdg.filter(pl.col("pdgid").is_in(top_particles))
    .group_by(["target", "pdgid"])
    .len()
    .with_columns(
        (pl.col("len") / pl.col("len").sum().over("target")).alias("fraction")
    )
)

# Pivot to perfectly align quarks and gluons in parallel columns
# This completely avoids the use of lists and IndexError
df_pivot = df_counts.pivot(
    on="target", index="pdgid", values="fraction"
).fill_null(0.0)

# Reorder the pivoted dataframe to strictly match the abundance order
df_pivot = df_pivot.filter(pl.col("pdgid").is_in(top_particles)).sort(
    pl.col("pdgid").map_elements(lambda p: top_particles.index(p), return_dtype=pl.Int64)
)

# Extract the final vectors ready for Matplotlib
frac_g = df_pivot.select(pl.nth(1)).to_numpy().flatten()  # target = 0 (Gluon)
frac_q = df_pivot.select(pl.nth(2)).to_numpy().flatten()  # target = 1 (Quark)

# ==============================================================================
# 2. RENDERING THE DISCRETE BAR PLOT
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(8.0, 5.5))

x_indexes = np.arange(len(top_particles))
bar_width = 0.35

# Draw the structured bars
ax.bar(
    x_indexes - bar_width / 2,
    frac_g,
    bar_width,
    color=G_COLOR,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
    label="Gluon Jets (Background)",
)
ax.bar(
    x_indexes + bar_width / 2,
    frac_q,
    bar_width,
    color=Q_COLOR,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
    label="Quark Jets (Signal)",
)

# Map the names in LaTeX format on the X axis
xtick_names = [pdg_labels.get(int(p), f"ID: {int(p)}") for p in top_particles]
ax.set_xticks(x_indexes)
ax.set_xticklabels(xtick_names, rotation=0, fontsize=12)

ax.set_xlabel("Particle Identity (PDG Code)", fontsize=13)
ax.set_ylabel("Relative Fraction per Jet Type", fontsize=13)
ax.set_title(
    "Jet Particle Composition (Hadronization Layer)",
    loc="left",
    fontweight="bold",
    fontsize=14,
)

# Style formatting details
ax.tick_params(direction="in", top=False, right=True, labelsize=11)
ax.grid(True, axis="y", linestyle=":", alpha=0.6)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=11)

plt.tight_layout()
plt.show()

# Strict memory cleanup
del (
    num_jets,
    max_multiplicity,
    _,
    pdg_labels,
    top_particles,
    frac_g,
    frac_q,
    fig,
    ax,
    df_pdg,
    df_counts,
    df_pivot,
    labels_flat,
    pdgid_flat
)
gc.collect()

From this plot we can't see any clear pattern or anomaly that may be present. The distributions of the particles in the Gluon jets and Quark jets are very similar, which is consistent with the fact that the two distributions are different. We will aproach this problem using a pondered electric charge, which is a measure of the charge of a jet based on the charges of the particles that make up the jet.

The other global feature is the pondered electric charge, which is a discrete variable. We can express as:

\begin{equation}
Q_{jet} =\frac{\sum_{i=1}^{N} q_i \cdot p_{T,i}^\kappa}{\left(\sum_{i=1}^{N} p_{T,i}\right)^\kappa}
\end{equation}

where $q_i$ is the electric charge of the $i$-th particle in the jet, $p_{T,i}$ is its transverse momentum, and $\kappa$ is a parameter that controls the weighting of the particles. In this case we use $\kappa = 0.5$.

we expect that the distribution of the pondered electric charge will be different for quark and gluon jets, since quarks have a non-zero electric charge while gluons are electrically neutral. In particular, we expect that the distribution of the pondered electric charge for quark jets will be centered around a non-zero value, while the distribution for gluon jets will be centered around zero.

In [ ]:
from matplotlib.gridspec import GridSpec
import pandas as pd

In [ ]:
# ==============================================================================
# 1. Electric Charge Extraction and Vectorized Preprocessing
# ==============================================================================
# Extracting kinematic and identity blocks (Shape: [100000, 139])
pt_block = X[:, :, 0]
pdgid_block = X[:, :, 3]

# Definition of the charge dictionary mapped to NumPy vectors
# We map the most abundant PDG IDs to their integer electric charges
charges_dict = {
    211: 1.0,  # pi+
    -211: -1.0,  # pi-
    111: 0.0,  # pi0
    22: 0.0,  # gamma
    321: 1.0,  # K+
    -321: -1.0,  # K-
    2212: 1.0,  # proton
    -2212: -1.0,  # antiproton
    11: -1.0,  # e-
    -11: 1.0,  # e+
    13: -1.0,  # mu-
    -13: 1.0,  # mu+
    2112: 0.0,  # neutron
    -2112: 0.0,  # antineutron
    130: 0.0,  # K_L^0
    310: 0.0,  # K_S^0
}

# Create the charge matrix, assigning 0.0 by default (suitable for padding and neutral particles)
charge_matrix = np.zeros_like(pdgid_block, dtype=np.float32)
for pdg_id, charge_val in charges_dict.items():
    charge_matrix[pdgid_block == pdg_id] = charge_val

# ==============================================================================
# 2. MATRIX CALCULATION OF JET CHARGE (Q_jet)
# ==============================================================================
kappa = 0.5

# Local pT weighting: (pT_i)^kappa
weighted_pt = np.power(pt_block, kappa)

# Global weighted sum for each jet: Sum_i (Q_i * (pT_i)^kappa)
numerator = np.sum(charge_matrix * weighted_pt, axis=1)

# Global denominator: (pT_jet)^kappa. We use the total sum of real pT per jet
pt_jet = np.sum(pt_block, axis=1)
pt_jet_safe = np.where(pt_jet == 0, 1.0, pt_jet)
denominator = np.power(pt_jet_safe, kappa)

# Jet Charge Final Calculation
q_jet = numerator / denominator

# Class Separation (0: Gluon, 1: Quark)
q_jet_q = q_jet[y == 1]
q_jet_g = q_jet[y == 0]

# Statistical metrics for the classes
mean_q, median_q = q_jet_q.mean(), np.median(q_jet_q)
mean_g, median_g = q_jet_g.mean(), np.median(q_jet_g)

# ==============================================================================
# 3. HYBRID CANVAS LAYOUT (GridSpec Vertical)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.06)
ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

#Q_COLOR = "#1f77b4"  # Blue for Quark (Signal)
#G_COLOR = "#e377c2"  # Pink/Mint for Gluon (Background)

# ==============================================================================
# 4.  UPPER PANEL: ADJUSTED DENSITIES (HISTOGRAMS)
# ==============================================================================
bins_qjet = np.linspace(-1.0, 1.0, 100)

ax_hist.hist(
    q_jet_q,
    bins=bins_qjet,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=rf"Quark Jets ($\mu={mean_q:.3f}$)",
)
ax_hist.hist(
    q_jet_g,
    bins=bins_qjet,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=rf"Gluon Jets ($\mu={mean_g:.3f}$)",
)

# Vertical lines indicating the means of the distributions
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(
    0.0, color="gray", linestyle=":", lw=1.0
)  # Reference for pure neutrality

ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    "Jet Charge Distribution: Quantum Flavor Asymmetry",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 5.  LOWER PANEL: VIOLINPLOT + BOXPLOT COMBINED
# ==============================================================================
df_box = pd.DataFrame(
    {
        "JetCharge": np.concatenate([q_jet_q, q_jet_g]),
        "Class": ["Quark (Signal)"] * len(q_jet_q) + ["Gluon (Bkg)"] * len(q_jet_g),
    }
)

# Smoothed background violinplot
sns.violinplot(
    x="JetCharge",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    bw_method="silverman",
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Narrow boxplot for exact quartiles
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}
flierprops = dict(marker=".", markersize=1.5, alpha=0.02, markeredgecolor="gray")

sns.boxplot(
    x="JetCharge",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    flierprops=flierprops,
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# Fixed annotation of absolute minimum and maximum at the inner extremes of the panel
ax_box.text(
    -0.75,
    -0.2,
    f"Quark Range: [{q_jet_q.min():.2f}, {q_jet_q.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=Q_COLOR,
    ha="left",
)
ax_box.text(
    -0.75,
    1.25,
    f"Gluon Range: [{q_jet_g.min():.2f}, {q_jet_g.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=G_COLOR,
    ha="left",
)

# Lower panel formatting
ax_box.set_xlabel(
    r"Weighted Jet Charge ($Q_{\mathrm{jet}}$ with $\kappa=0.5$)", fontsize=13
)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(-1.0, 1.0)  # Optimal symmetric centering for U(1) physics
ax_box.grid(True, linestyle=":", alpha=0.5)

# Hide X-axis labels of the upper histogram to avoid typographic overlap
plt.setp(ax_hist.get_xticklabels(), visible=False)

# Collider metadata (HEP Style)
fig.suptitle(
    r"Pythia 8.2 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Kinematic Window: $p_T^{jet} \in [500, 550]$ GeV, Pondered by Local $p_T$ Fractions",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

# ==============================================================================
# 6. RIGOROUS RAM CLEANUP
# ==============================================================================
del (
    pt_block,
    pdgid_block,
    charges_dict,
    charge_matrix,
    weighted_pt,
    numerator,
    pt_jet,
    pt_jet_safe,
    denominator,
    q_jet,
)
del (
    #q_jet_q,
    #q_jet_g,
    df_box,
    kappa,
    mean_q,
    mean_g,
    median_q,
    median_g
)

del (
    fig,
    gs,
    ax_hist,
    ax_box,
    bins_qjet,
    meanprops,
    flierprops
)
gc.collect()

This pondered electric charge could not be a good feature to distinguish between quark and gluon jets, since the quark jets have a non-zero electric charge while the gluon jets are electrically neutral, but the diference in the distribution of the pondered electric charge is not significant enough to be used as a feature for classification only based in the plot. Now we will apply a statistical test to see if the difference is significant or not. We will use the Kolmogorov-Smirnov test, which is a non-parametric test that can be used to compare the distributions of two samples. The Kolmogorov-Smirnov test compares the empirical cumulative distribution functions of the two samples, while the Wasserstein distance measures the distance between the two distributions in terms of the amount of "work" required to transform one distribution into the other. 

- $H_0$: The two samples are drawn from the same distribution.
- $H_1$: The two samples are drawn from different distributions.

The p-value expected is very low, this is the reason to use a sigma threshold to reject the null hypothesis. We will use a threshold of 5 sigma, which corresponds to a p-value of $3.5 \times 10^{-7}$. This means that if the p-value is less than this threshold, we will reject the null hypothesis and conclude that the two samples are drawn from different distributions.

We will also calculate the Wasserstein distance, which is a measure of the distance between two probability distributions. The Wasserstein distance is defined as the minimum amount of "work" required to transform one distribution into the other, where "work" is defined as the amount of probability mass that needs to be moved and the distance it needs to be moved. The Wasserstein distance is a useful measure of the difference between two distributions, since it takes into account both the shape and the location of the distributions.

Also we will calculate the skewness of the gluon distribution, which is a measure of the asymmetry of the distribution. The skewness is defined as the third standardized moment of the distribution, and it can be used to determine if the distribution is symmetric or not. A skewness of 0 indicates a symmetric distribution, while a positive skewness indicates a distribution that is skewed to the right, and a negative skewness indicates a distribution that is skewed to the left. We will also calculate the p-value associated with the skewness test, which tests the null hypothesis that the distribution is symmetric. If the p-value is less than 0.05, we will reject the null hypothesis and conclude that the distribution is not symmetric.

In [ ]:
import numpy as np
import scipy.stats as stats

In [ ]:
# ==============================================================================
# Statistical Significance Tests for Pondered Jet Charge Distributions
# ==============================================================================

# 1. Kolmogorov-Smirnov Test (Sensitive to differences in shape and width)
ks_stat, p_val_ks = stats.ks_2samp(q_jet_q, q_jet_g)

# Convert p-value to Sigmas (Z-score) safely against memory underflows
if p_val_ks == 0.0:
    # If the p-value is smaller than the 64-bit float limit (2.2e-308)
    sigmas_ks = ">>> 50 sigma (Truncated by numerical precision)"
else:
    sigmas_ks = f"{stats.norm.ppf(1 - p_val_ks / 2):.2f} sigma"

# 2. Wasserstein Distance (Earth Mover's Distance)
# Quantifies the "work" needed to transform the gluon shape into the quark shape
wasserstein_dist = stats.wasserstein_distance(q_jet_q, q_jet_g)

# 3. Skewness Test for Gluon Calibration
skew_gluon = stats.skew(q_jet_g)
p_val_skew_g = stats.skewtest(q_jet_g).pvalue

# ==============================================================================
# FORMAL REPORT OF RESULTS
# ==============================================================================
print("=" * 80)
print(
    f"STATISTICAL SIGNIFICANCE ANALYSIS:  PONDERED JET CHARGE RECONSTRUCTION (kappa = 0.5)"
)
print("=" * 80)
print(f"[-] Kolmogorov-Smirnov D Statistic: {ks_stat:.4f}")
print(f"[-] p-value associated with KS test: {p_val_ks}")
print(f"[-] STATISTICAL SIGNIFICANCE (Z)      : {sigmas_ks}")
print("-" * 80)
print(
    f"[-] Wasserstein Distance (EMD)        : {wasserstein_dist:.4f} GeV^(1/2)"
)
print("-" * 80)
print(f"[-] Gluon Symmetry Calibration        :")
print(f"    * Empirical Skewness             : {skew_gluon:.4f}")
print(f"    * p-value for pure symmetry (H0) : {p_val_skew_g:.4f}")
print("=" * 80)

With this results we can conclude that the K-S test do reject the null hypotesis but the statistical significance (Z) shows and inf sigma, both results are probably caused by the large number of samples. the D statistic, 0.1023, means that the maximum difference between the two empirical cumulative distribution functions is 0.1023, considering that the range of the pondered electric charge is between -1 and 1, this could be a slight difference. The p-value associated with the K-S test is very low, which means that we reject the null hypothesis and conclude that the two distributions are different.

The Wasserstein distance is 0.0754 shows how much "work" is required to transform one distribution into the other, which is a measure of the difference between the two distributions. This difference is not very large, but it is still significant enough to be considered a difference between the two distributions.

The skewness of the gluon distribution is very close to 0, the p-value associated with the skewness test is greater than 0.05, which means that we fail to reject the null hypothesis and conclude that the distribution is symmetric. This is important because it means that the distribution of the pondered electric charge for gluon jets is symmetric, around 0, and the difference between the two distributions is could be used to distinguish between quark and gluon jets, but with a lower statistical significance.

### Local features

For local features we will explore the distribution of the transverse momentum. Pseudorapidity, and azimuthal angle for both quark and gluon jets are expected to be similar, but we will see if there are any differences in the distributions.

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import seaborn as sns

In [ ]:
# 1. Style configuration
plt.style.use("seaborn-v0_8-whitegrid")

# 2. Separate real constituents by class for comparison
# Avoid copying the entire DataFrame, only select the necessary column
quarks_particles = df_reals.filter(pl.col("target") == 1)
gluons_particles = df_reals.filter(pl.col("target") == 0)

# 3. Initialize the Canvas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Consistent color palette (Blue for Quarks / Orange-Red for Gluons)
#colors = {"quark": "#1f77b4", "gluon": "#e377c2"}

# --- Subplot 1: Pseudorapidity (eta) Distribution ---
ax0 = axes[0]
sns.histplot(
    data=quarks_particles,
    x="eta",
    stat="density",
    bins=100,
    color=Q_COLOR,
    element="step",
    fill=True,
    alpha=0.3,
    label="Quark Jets (Signal)",
    ax=ax0,
)
sns.histplot(
    data=gluons_particles,
    x="eta",
    stat="density",
    bins=100,
    color=G_COLOR,
    element="step",
    fill=False,
    alpha=0.8,
    label="Gluon Jets (Bkg)",
    ax=ax0,
)

ax0.set_title("Detector Acceptance & Jet Kinematics", fontweight="bold")
ax0.set_xlabel(r"Constituent Pseudorapidity ($\eta$)", fontsize=14)
ax0.set_ylabel("Normalized Density", fontsize=14)
ax0.set_xlim([-2.5, 2.5])  # Un poco más allá del corte nominal de |y| < 1.7
ax0.axvline(
    1.7, color="black", linestyle="--", alpha=0.5, label="Nominal Jet Cut (|y| < 1.7)"
)
ax0.axvline(-1.7, color="black", linestyle="--", alpha=0.5)
ax0.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=11)

# --- Subplot 2: Azimuthal Angle (phi) Distribution ---
ax1 = axes[1]
sns.histplot(
    data=quarks_particles,
    x="phi",
    stat="density",
    bins=50,
    color=Q_COLOR,
    element="step",
    fill=True,
    alpha=0.3,
    label="Quark Jets",
    ax=ax1,
)
sns.histplot(
    data=gluons_particles,
    x="phi",
    stat="density",
    bins=50,
    color=G_COLOR,
    element="step",
    fill=False,
    alpha=0.8,
    label="Gluon Jets",
    ax=ax1,
)

ax1.set_title("Azimuthal Isotropy Verification", fontweight="bold")
ax1.set_xlabel(r"Constituent Azimuthal Angle ($\phi$ [rad])", fontsize=14)
ax1.set_ylabel("Normalized Density", fontsize=14)
#ax1.set_xlim([-np.pi - 0.2, np.pi + 0.2])
ax1.set_xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax1.set_xticklabels([r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"])
ax1.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=11)

# --- Global formatting details ---
for ax in axes:
    ax.tick_params(direction="in", top=True, right=True, length=5)
    ax.grid(True, linestyle=":", alpha=0.6)

plt.suptitle(
    "Absolute Coordinate Distributions",
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# 4. Strict RAM cleanup
del (
    quarks_particles,
    gluons_particles,
    fig,
    axes,
    ax0,
    ax1
    )
gc.collect()

Now we can see why the mean value of $\phi$ is close to $\pi$, because the range of $\phi$ is from $0$ to $2\pi$, and the distribution is symmetric around $\pi$. The distribution of $\phi$ and $\eta$ are similar for both quark and gluon jets, this confirms that the jets are isotropic, as expected.

Let's calculate the center of each jet and see the dispersión in the $\phi vs \eta$ space.

In [ ]:
# 1. Calculate the pT-weighted center for each jet
jet_centers = df_reals.group_by("jet_id").agg(
    [
        ((pl.col("eta") * pl.col("pt")).sum()
        / pl.col("pt").sum()).alias("jet_eta"),
        ((pl.col("phi") * pl.col("pt")).sum()
        / pl.col("pt").sum()).alias("jet_phi"),
    ]
)

# 2. Join the jet centers back to the dataframe of charged particles
df_relatives = df_reals.join(jet_centers, on="jet_id")

# 3. Calculate the relative deltas
# Note: For delta_phi, we need to ensure the periodicity of the detector cylinder [-pi, pi]
df_relatives = df_relatives.with_columns(
    [
        (pl.col("eta") - pl.col("jet_eta")).alias("delta_eta"),
        # Angular correction to avoid jumps at the phi boundaries
        (
            (pl.col("phi") - pl.col("jet_phi") + np.pi) % (2 * np.pi) - np.pi
        ).alias("delta_phi"),
    ]
)

del jet_centers
gc.collect()

In [ ]:
import matplotlib.colors as colors
from matplotlib.colors import LogNorm

In [ ]:
# Separate the deltas by class for the 2D histogram
quarks_df = df_relatives.filter(pl.col("target") == 1)
gluons_df = df_relatives.filter(pl.col("target") == 0)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True, sharey=True)

# Adjust the range to the actual jet size (R = 0.4) plus a small margin
img_range = [[-0.6, 0.6], [-0.6, 0.6]]
bin_resolution = 120

# --- Panel 1: Quarks (Signal) ---
im0 = axes[0].hist2d(
    quarks_df["delta_eta"].to_numpy(),
    quarks_df["delta_phi"].to_numpy(),
    bins=bin_resolution,
    range=img_range,
    cmap="inferno",
    norm=LogNorm(),
)
cbar0 = fig.colorbar(im0[3], ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label("Constituent Density (Log Scale)", rotation=270, labelpad=15)
axes[0].set_title(
    "Signal: Quark Jets", fontsize=14, fontweight="bold"
)
axes[0].set_xlabel(r"$\Delta \eta = \eta_{part} - \eta_{jet}$", fontsize=14)
axes[0].set_ylabel(r"$\Delta \phi = \phi_{part} - \phi_{jet}$", fontsize=14)

# --- Panel 2: Gluons (Background) ---
im1 = axes[1].hist2d(
    gluons_df["delta_eta"].to_numpy(),
    gluons_df["delta_phi"].to_numpy(),
    bins=bin_resolution,
    range=img_range,
    cmap="inferno",
    norm=LogNorm(),
)
cbar1 = fig.colorbar(im1[3], ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label("Constituent Density (Log Scale)", rotation=270, labelpad=15)
axes[1].set_title(
    "Background: Gluon Jets", fontsize=14, fontweight="bold"
)
axes[1].set_xlabel(r"$\Delta \eta = \eta_{part} - \eta_{jet}$", fontsize=14)

# Strict formatting for both panels
for ax in axes:
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(img_range[0])
    ax.set_ylim(img_range[1])
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle(
    "Jet Substructure: Relative Phase Space Geometry ($q/g$ Separation)",
    fontsize=18,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

del quarks_df, gluons_df
gc.collect()

In [ ]:
import gc
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from matplotlib.gridspec import GridSpec

In [ ]:
# ==============================================================================
# 1. EXTRACCIÓN Y CÁLCULO VECTORIZADO DE DELTA R
# ==============================================================================
# Extraemos las matrices de desviaciones de tu DataFrame 'df_relatives'
# Nota: Aplanamos para analizar la distribución total de todos los constituyentes
deta_flat = df_relatives["delta_eta"].to_numpy()
dphi_flat = df_relatives["delta_phi"].to_numpy()
target_flat = df_relatives["target"].to_numpy()

# Cálculo euclidiano de la separación angular de cada partícula respecto al eje
dr_flat = np.sqrt(deta_flat**2 + dphi_flat**2)

# Segregación por clases usando las etiquetas reales (0: Gluón, 1: Quark)
dr_q = dr_flat[target_flat == 1]
dr_g = dr_flat[target_flat == 0]

# Métricas estadísticas descriptivas para las marcas e inyección de texto
mean_q, median_q = dr_q.mean(), np.median(dr_q)
mean_g, median_g = dr_g.mean(), np.median(dr_g)

# ==============================================================================
# 2. CONFIGURACIÓN DEL CANVAS HÍBRIDO (GridSpec Vertical)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

# División 75/25 para el histograma superior y los diagramas de caja inferiores
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.06)
ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# Colores consistentes con tu pipeline de subestructura
#Q_COLOR = "#1f77b4"  # Azul para Quark (Signal)
#G_COLOR = "#e377c2"  # Rosa/Menta para Gluón (Background)

# ==============================================================================
# 3. PANEL SUPERIOR: DENSIDADES DE PASOS (HISTOGRAMAS 1D)
# ==============================================================================
# Límites acotados físicamente por el radio del algoritmo anti-kt (R=0.4) + margen
bins_dr = np.linspace(0.0, 0.55, 110)

ax_hist.hist(
    dr_q,
    bins=bins_dr,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=rf"Quark Jets ($\mu={mean_q:.3f}$)",
)
ax_hist.hist(
    dr_g,
    bins=bins_dr,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=rf"Gluon Jets ($\mu={mean_g:.3f}$)",
)

# Líneas discontinuas indicadoras de los valores medios
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)

ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    r"Jet Substructure: Radial Constituent Dispersion ($\Delta R$)",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 4. PANEL INFERIOR: VIOLINPLOT + BOXPLOT HORIZONTAL CONTRAPUESTO
# ==============================================================================
df_box = pd.DataFrame(
    {
        "DeltaR": np.concatenate([dr_q, dr_g]),
        "Class": ["Quark (Signal)"] * len(dr_q) + ["Gluon (Bkg)"] * len(dr_g),
    }
)

# Capa inferior: Violinplot de densidad suavizada
sns.violinplot(
    x="DeltaR",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Capa superior: Boxplot angosto optimizado (showfliers=False para evitar PDF pesado)
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}

sns.boxplot(
    x="DeltaR",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    showfliers=False,  # Evita que miles de puntos vectoriales inflen el peso del archivo
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# Anotación fija de mínimos y máximos en el extremo superior interno
ax_box.text(
    0.02,
    -0.2,
    f"Quark Range: [{dr_q.min():.2f}, {dr_q.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=Q_COLOR,
    ha="left",
)
ax_box.text(
    0.02,
    0.85,
    f"Gluon Range: [{dr_g.min():.2f}, {dr_g.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=G_COLOR,
    ha="left",
)

# Formato estricto del eje compartido
ax_box.set_xlabel(
    r"Constituent Angular Distance to Jet Axis ($\Delta R = \sqrt{\Delta\eta^2 + \Delta\phi^2}$)",
    fontsize=13,
)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(-0.02, 0.55)  # Acotado al tamaño del cono real
ax_box.grid(True, linestyle=":", alpha=0.5)

# Ocultar etiquetas repetitivas del eje X superior
plt.setp(ax_hist.get_xticklabels(), visible=False)

# Metadatos del experimento (Estilo Publicación HEP)
fig.suptitle(
    r"Pythia 8.2 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Kinematic Selection: $p_T^{jet} \in [500, 550]$ GeV, All Resolved Constituents",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. LIMPIEZA TOTAL DE MEMORIA RAM
# ==============================================================================
del deta_flat, dphi_flat, target_flat, dr_flat, dr_q, dr_g, df_box
del fig, gs, ax_hist, ax_box, bins_dr, meanprops
gc.collect()

in the plot we can see an aparent difference. The quark jets are more concentrated in the center, while the gluon jets are more spread out. This is consistent with the fact that quark jets are more collimated than gluon jets, which means that they have a smaller angular spread and therefore a lower multiplicity.

Now, we are going to focus on the pt. We will order the particles by pt and see where the distribution completes the 80 % of the total pt. This will give us an idea of how the pt is distributed among the particles in the jet, and also a decision boundary to select the particles that contribute the most to the jet pt. We will see if there are any differences between quark and gluon jets in this regard.

In [ ]:
# ==============================================================================
# 1. pt sorting
# ==============================================================================
# Extract the pT component (100000, 139)
pt_matrix = X[:, :, 0]

# np.argsort gives indices from smallest to largest. With [:, ::-1] we invert to descending
sort_indices = np.argsort(pt_matrix, axis=1)[:, ::-1]

# Reorder the original X matrix in its 4 features using advanced indexing
# np.arange(X.shape[0])[:, None] ensures that each jet keeps its own particles
X_sorted = X[np.arange(X.shape[0])[:, None], sort_indices, :]

# Extract the sorted pT block for Pareto analysis
pt_block = X_sorted[:, :, 0]

# ==============================================================================
# 2. PARETO KINEMATIC PROCESSING
# ==============================================================================
# Cumulative sum along the particle index (column axis)
cumulative_pt = np.cumsum(pt_block, axis=1)

# The last cumulative element (index -1) represents the total recovered pT of the Jet
total_pt = cumulative_pt[:, -1]
total_pt_safe = np.where(total_pt == 0, 1.0, total_pt)  # Avoid division by zero

# Convert to cumulative fractions (scale from 0 to 1)
pt_fraction_matrix = cumulative_pt / total_pt_safe[:, None]

# Segregate profiles using the true labels 'y' (0: gluon, 1: quark)
quark_profiles = pt_fraction_matrix[y == 1]
gluon_profiles = pt_fraction_matrix[y == 0]

# Calculate the mean and standard deviation for each step of the sequence
mean_quark = np.mean(quark_profiles, axis=0)
std_quark = np.std(quark_profiles, axis=0)

mean_gluon = np.mean(gluon_profiles, axis=0)
std_gluon = np.std(gluon_profiles, axis=0)

# The X axis will go from particle 1 to the maximum (139)
max_particles = X.shape[1]
particle_indices = np.arange(1, max_particles + 1)

# ==============================================================================
# 3. AUTOMATIC TRUNCATION CALCULATION (80% THRESHOLD)
# ==============================================================================
# Find the first index where BOTH curves exceed 80% (0.80)
both_exceed_80 = (mean_quark >= 0.80) & (mean_gluon >= 0.80)

if np.any(both_exceed_80):
    N_cut = int(np.where(both_exceed_80)[0][0] + 1)
else:
    N_cut = max_particles

cut_fraction_quark = mean_quark[N_cut - 1]
cut_fraction_gluon = mean_gluon[N_cut - 1]

# ==============================================================================
# 4. PLOT CONSTRUCTION AND RENDERING (Publication Style)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(7.5, 5.5))

# Consistent color definitions
#QUARK_COLOR = "#1f77b4"  # Blue for signal
#GLUON_COLOR = "#e377c2"  # Pink/Mint for background

# Signal Curve (Quarks) with its uncertainty band
ax.plot(
    particle_indices,
    mean_quark,
    color=Q_COLOR,
    lw=2.5,
    label=r"Quark Jets (Signal)",
)
ax.fill_between(
    particle_indices,
    mean_quark - std_quark,
    mean_quark + std_quark,
    color=Q_COLOR,
    alpha=0.15,
    step="mid",
)

# Background Curve (Gluons) with its uncertainty band
ax.plot(
    particle_indices,
    mean_gluon,
    color=G_COLOR,
    lw=2.5,
    label="Gluon Jets (Background)",
    linestyle="--",
)
ax.fill_between(
    particle_indices,
    mean_gluon - std_gluon,
    mean_gluon + std_gluon,
    color=G_COLOR,
    alpha=0.12,
    step="mid",
)

# --- TRUNCATION THRESHOLD INDICATORS ---
ax.axhline(y=0.80, color="gray", linestyle=":", lw=1.0, alpha=0.8)
ax.axvline(
    x=N_cut,
    color="red",
    linestyle="-.",
    lw=1.5,
    label=f"Pareto Target ($N_{{cut}}={N_cut}$)",
)

ax.plot(N_cut, cut_fraction_quark, marker="o", color=Q_COLOR, markersize=7)
ax.plot(N_cut, cut_fraction_gluon, marker="o", color=G_COLOR, markersize=7)

# Explanatory text box with the actual retained percentages
physics_text = (
    f"Energy Retained at $N={N_cut}$:\n"
    f"Quarks: {cut_fraction_quark*100:.1f}%\n"
    f"Gluons: {cut_fraction_gluon*100:.1f}%"
)
ax.text(
    N_cut + 2,
    0.45,
    physics_text,
    fontsize=10,
    bbox=dict(
        facecolor="white", alpha=0.9, edgecolor="gray", boxstyle="round,pad=0.5"
    ),
)

# --- AXIS FORMATTING ---
ax.set_xlabel("Sorted Constituent Index ($i$)", fontsize=13)
ax.set_ylabel(
    r"Cumulative $p_T$ Fraction ($\sum p_{T,i} / p_{T}^{\mathrm{jet}}$)",
    fontsize=13
)
ax.set_xlim(1, 60)
ax.set_ylim(0.0, 1.05)

ax.tick_params(direction="in", top=True, right=True, labelsize=11)
ax.set_title(
    "Constituent Energy Scaling & Truncation Optimization",
    loc="left",
    fontweight="bold",
    fontsize=14
)
fig.suptitle(
    r"Pythia 8 Simulation, anti-$k_t$ ($R=0.4$), $p_T^{jet} \in [500, 550]$ GeV",
    fontsize=10.5,
    y=0.98,
    style="italic"
)

ax.grid(True, linestyle=":", alpha=0.5)
ax.legend(frameon=True, facecolor="white", edgecolor="none", loc="lower right")

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. MEMORY CLEANUP
# ==============================================================================
del (
    cumulative_pt,
    total_pt,
    total_pt_safe,
    pt_fraction_matrix,
    quark_profiles,
    gluon_profiles,
)
del mean_quark, std_quark, mean_gluon, std_gluon, particle_indices, sort_indices
gc.collect()

This plot shows the cumulative distribution of the transverse momentum for both quark and gluon jets. We can see that the quark jets have a steeper slope, which means that they have a higher fraction of their total pt concentrated in the first few particles. This is consistent with the fact that quark jets are more collimated than gluon jets, which means that they have a smaller angular spread and therefore a lower multiplicity.

Other conclusion from this plot is the point where the cumulative distribution reaches 80 % of the total pt. To mantain the dimensionality of the input data, we will select the first 15 particles, which is a good compromise between retaining most of the pt and reducing the number of particles. This will also help to reduce the computational cost of training the model.

In [ ]:
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
import pandas as pd

In [19]:
# ==============================================================================
# 1. EXTRACCIÓN DE DATOS Y ESTADÍSTICA DESCRIPTIVA CON POLARS
# ==============================================================================
# Calcular la multiplicidad real por jet (con pt > 0)
df_mult = df_reals.group_by(["jet_id", "target"]).agg(
    pl.len().alias("multiplicity")
)

# Separar en arrays de NumPy por clase
mult_q = df_mult.filter(pl.col("target") == 1)["multiplicity"].to_numpy()
mult_g = df_mult.filter(pl.col("target") == 0)["multiplicity"].to_numpy()

# Extracción de métricas clave para la inyección de texto y marcas
min_q, max_q = mult_q.min(), mult_q.max()
min_g, max_g = mult_g.min(), mult_g.max()
mean_q, median_q = mult_q.mean(), np.median(mult_q)
mean_g, median_g = mult_g.mean(), np.median(mult_g)

# ==============================================================================
# 2. CONFIGURACIÓN DEL CANVAS HÍBRIDO (GridSpec)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

# Definición de las proporciones: 3 partes para histograma, 1 para boxplot
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.00)

ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# Paleta de colores consistente
#Q_COLOR = "#1f77b4"  # Azul (Signal - Quark)
#G_COLOR = "#e377c2"  # Rosa/Menta (Background - Gluon)

# ==============================================================================
# 3. PANEL SUPERIOR: HISTOGRAMAS DE PASOS (STEP)
# ==============================================================================
bins_mult = np.arange(0, max_multiplicity + 2, 2)

ax_hist.hist(
    mult_q,
    bins=bins_mult,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=f"Quark Jets ($\mu={mean_q:.1f}$)",
)
ax_hist.hist(
    mult_g,
    bins=bins_mult,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=f"Gluon Jets ($\mu={mean_g:.1f}$)",
)

# Líneas de las medias
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)

# LÍNEA CRUCIAL: Tu propuesta de corte N = 16
N_propuesto = 16
ax_hist.axvline(
    N_propuesto,
    color="red",
    linestyle="-.",
    lw=1.8,
    label=f"Pareto Cut Target ($N={N_propuesto}$)",
)

# Formato del panel superior
ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    "Jet Substructure: Multiplicity Dispersions & Padding Analysis",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 4. PANEL INFERIOR: BOXPLOTS Y VIOLINPLOTS HORIZONTALES HÍBRIDOS
# ==============================================================================
# Preparar DataFrame para Seaborn
df_box = pd.DataFrame(
    {
        "Multiplicity": np.concatenate([mult_q, mult_g]),
        "Class": ["Quark (Signal)"] * len(mult_q) + ["Gluon (Bkg)"] * len(mult_g),
    }
)

# Capa de Violín (Densidad suavizada de fondo)
sns.violinplot(
    x="Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    bw_method="silverman",
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Capa de Boxplot Estrecho (Cuartiles exactos)
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}
flierprops = dict(marker=".", markersize=1.5, alpha=0.05, markeredgecolor="gray")

sns.boxplot(
    x="Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    flierprops=flierprops,
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# --- MARCAS DE MÍNIMOS Y MÁXIMOS ABSOLUTOS ---
ax_box.text(
    3.5,
    -0.2,
    f"Quark Range: [{min_q}, {max_q}]",
    fontsize=9,
    fontweight="bold",
    color=Q_COLOR,
    ha="left",
)
ax_box.text(
    3.5,
    1.25,
    f"Gluon Range: [{min_g}, {max_g}]",
    fontsize=9,
    fontweight="bold",
    color=G_COLOR,
    ha="left",
)

# Replicar la línea de N=16 en el boxplot para ver el choque visual con las cajas
ax_box.axvline(N_propuesto, color="red", linestyle="-.", lw=1.8, alpha=0.7)

# Formato del panel inferior
ax_box.set_xlabel("Jet Particle Multiplicity (Constituent Count)", fontsize=13)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(0, 100)  # Enfocado para ver la física, omitiendo outliers extremos aislados
ax_box.grid(True, linestyle=":", alpha=0.5)

plt.setp(ax_hist.get_xticklabels(), visible=False)  # Ocultar etiquetas X del histograma superior

# --- METADATOS ESTILO HEP ---
fig.suptitle(
    r"Pythia 8.226 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Jet Selection Criteria: $p_T^{jet} \in [500, 550]$ GeV, $|y^{jet}| < 1.7$",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. LIMPIEZA DE MEMORIA RAM
# ==============================================================================
del df_mult, df_box, mult_q, mult_g
gc.collect()

## Filters and selection of the particles

After this analysis, we can conclude that the quark and gluon jets have different distributions of their features, which can be used to distinguish between them. The quark jets are more collimated and have a higher fraction of their total pt concentrated in the first few particles, while the gluon jets are more spread out and have a lower fraction of their total pt concentrated in the first few particles. This information can be used to design a model that can effectively distinguish between quark and gluon jets based on their features.

The multiplicity could also be a great feature to distinguish between quark and gluon jets, since the quark jets have a lower multiplicity than the gluon jets. The pondered electric charge could not be a good feature to distinguish between quark and gluon jets, since the quark jets have a non-zero electric charge while the gluon jets are electrically neutral, but the diference in the distribution of the pondered electric charge is not significant enough to be used as a feature for classification.

In [ ]:
import gc
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Diccionario de cargas del Particle Data Group (PDG)
CHARGES_DICT = {
    211: 1.0, -211: -1.0, 111: 0.0, 22: 0.0, 
    321: 1.0, -321: -1.0, 2212: 1.0, -2212: -1.0, 
    11: -1.0, -11: 1.0, 13: -1.0, -13: 1.0, 
    2112: 0.0, -2112: 0.0, 130: 0.0, 310: 0.0
}

def features_qg(X, y, kappa=0.5, p_pareto=0.80):
    """
    Réplica exacta del pipeline optimizado. Diseñado para retornar tanto la
    matriz final para la KAN como las variables intermedias para fines de diagnóstico en el Notebook.
    """
    n_events, max_particles, n_features = X.shape
    print(f"--- Iniciando Análisis Exploratorio ---")
    print(f"Dimensión de entrada: X = {X.shape}, y = {y.shape}")

    # -------------------------------------------------------------------------
    # PASO 1: Filtrado Cinemático y Geométrico (Jet Clean-up)
    # -------------------------------------------------------------------------
    pt_raw = X[:, :, 0]
    eta_raw = X[:, :, 1]
    phi_raw = X[:, :, 2]
    pdg_raw = X[:, :, 3]

    sum_pt = np.sum(pt_raw, axis=1)
    sum_pt_safe = np.where(sum_pt == 0, 1.0, sum_pt)

    # Coordenadas del baricentro del Jet
    eta_jet = np.sum(pt_raw * eta_raw, axis=1) / sum_pt_safe
    
    # Manejo robusto del borde detector (-pi, pi)
    sin_phi_avg = np.sum(pt_raw * np.sin(phi_raw), axis=1) / sum_pt_safe
    cos_phi_avg = np.sum(pt_raw * np.cos(phi_raw), axis=1) / sum_pt_safe
    phi_jet = np.arctan2(sin_phi_avg, cos_phi_avg)

    # Distancia Radial R
    d_eta = eta_raw - eta_jet[:, None]
    d_phi = np.arctan2(np.sin(phi_raw - phi_jet[:, None]), np.cos(phi_raw - phi_jet[:, None]))
    d_R = np.sqrt(d_eta**2 + d_phi**2)

    # Máscara infrarroja y geométrica (Cono R=0.4)
    unphysical_mask = (pt_raw <= 1e-3) | (d_R > 0.4)
    
    # Limpieza estricta In-Place
    X[unphysical_mask] = 0.0
    d_R[unphysical_mask] = 0.0

    # -------------------------------------------------------------------------
    # PASO 2: Extracción de Características Globales
    # -------------------------------------------------------------------------
    multiplicity = np.sum(X[:, :, 0] > 0.0, axis=1).astype(np.float32)

    # Mapeo vectorizado de cargas
    max_pdg = int(np.max(np.abs(pdg_raw))) if np.max(np.abs(pdg_raw)) > 0 else 2212
    charge_lookup = np.zeros(max_pdg + 1, dtype=np.float32)
    # (Corrección de signo)
    for pdg, chg in CHARGES_DICT.items():
        if abs(pdg) <= max_pdg:
            # 1. Almacenamos el valor absoluto de la carga en la tabla de búsqueda
            charge_lookup[abs(pdg)] = abs(chg) 

    # 2. Reconstruimos recuperando el signo original del PDG ID de cada constituyente
    charge_matrix = charge_lookup[np.abs(pdg_raw).astype(np.int32)] * np.sign(pdg_raw)
    
    pt_weighted = np.power(X[:, :, 0], kappa)
    numerator = np.sum(charge_matrix * pt_weighted, axis=1)
    
    pt_jet = np.sum(X[:, :, 0], axis=1)
    pt_jet_safe = np.where(pt_jet == 0, 1.0, pt_jet)
    denominator = np.power(pt_jet_safe, kappa)
    
    q_jet = numerator / denominator

    # -------------------------------------------------------------------------
    # PASO 3: Ordenamiento Cinemático y Análisis de Pareto
    # -------------------------------------------------------------------------
    sorted_indices = np.argsort(-X[:, :, 0], axis=1)
    row_indices = np.arange(n_events)[:, None]

    X = X[row_indices, sorted_indices, :]
    d_R = d_R[row_indices, sorted_indices]

    # Curva de energía acumulada
    pt_cumsum = np.cumsum(X[:, :, 0], axis=1)
    pt_frac_cumsum = pt_cumsum / np.where(pt_jet[:, None] <= 0, 1.0, pt_jet[:, None])

    # Encontrar dinámicamente dónde se cruza el umbral (por ejemplo, 80%)
    idx_threshold = np.argmax(pt_frac_cumsum >= p_pareto, axis=1) + 1
    n_cut = int(np.percentile(idx_threshold, 90))
    n_cut = max(n_cut, 5) # Piso mínimo de partículas constitutivas

    print(f"Corte óptimo de Pareto determinado ({int(p_pareto*100)}% energía al percentil 90): N_cut = {n_cut}")

    # Truncamiento de subestructura
    X_truncated = X[:, :n_cut, :]
    d_R_truncated = d_R[:, :n_cut]

    # -------------------------------------------------------------------------
    # PASO 4: Construcción y Escalamiento KAN
    # -------------------------------------------------------------------------
    z_effective = X_truncated[:, :, 0] / pt_jet_safe[:, None]
    z_effective[X_truncated[:, :, 0] <= 0.0] = 0.0

    # Diccionario de escaladores locales (Ajustados con este set)
    scalers = {
        'z_max': float(np.max(z_effective)) if np.max(z_effective) > 0 else 1.0,
        'q_max': float(np.max(np.abs(q_jet))) if np.max(np.abs(q_jet)) > 0 else 1.0,
        'n_min': float(np.min(multiplicity)),
        'n_max': float(np.max(multiplicity)) if np.max(multiplicity) > 0 else 1.0
    }

    z_scaled = z_effective / scalers['z_max']
    dr_scaled = d_R_truncated / 0.4
    q_scaled = q_jet / scalers['q_max']
    
    n_scaled = 2.0 * ((multiplicity - scalers['n_min']) / (scalers['n_max'] - scalers['n_min'])) - 1.0

    # Construcción de la matriz final entrelazada
    processed_matrix = np.zeros((n_events, 2 + 2 * n_cut), dtype=np.float32)
    processed_matrix[:, 0] = n_scaled
    processed_matrix[:, 1] = q_scaled
    processed_matrix[:, 2::2] = z_scaled
    processed_matrix[:, 3::2] = dr_scaled

    print(f"Dimensión de salida del tensor final KAN: {processed_matrix.shape}")
    print(f"Valores NaN detectados: {np.isnan(processed_matrix).any()} | Inf: {np.isinf(processed_matrix).any()}")

    # Retornamos los diccionarios y variables intermedias estructuradas para graficar
    diagnostics = {
        'multiplicity_raw': multiplicity,
        'q_jet_raw': q_jet,
        'pt_frac_cumsum': pt_frac_cumsum,
        'n_scaled': n_scaled,
        'q_scaled': q_scaled,
        'z_scaled': z_scaled,
        'dr_scaled': dr_scaled
    }

    return processed_matrix, diagnostics

In [ ]:
import gc
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
try:
    with np.load('../data/raw/quark-gluon/QG_jets_fp32_0.npz', 'r') as f:
        print("File keys:", list(f.keys()), "\n")

        for key in f.keys():
            print(f"Shape of '{key}': {f[key].shape}")

        X = f['X'][:10000, :, :]
        y = f['y'][:10000]
except Exception as e:
    print(f"Error loading data using numpy: {e}")

In [ ]:
processed_matrix, diagnostics = features_qg(X, y, kappa=0.5, p_pareto=0.80)

In [ ]:
import gc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

In [ ]:
# -------------------------------------------------------------------------
# 1. EXTRACCIÓN DIRECTA DE LA VARIABLE GLOBAL NORMALIZADA
# -------------------------------------------------------------------------
# Columna 0 corresponde estrictamente a N_scaled en el rango [-1, 1]
n_scaled = processed_matrix[:, 0]

# Segmentación de alta velocidad por máscaras booleanas
mult_q_scaled = n_scaled[y == 1]
mult_g_scaled = n_scaled[y == 0]

# Métricas estadísticas descriptivas en el espacio transformado
mean_q, median_q = mult_q_scaled.mean(), np.median(mult_q_scaled)
mean_g, median_g = mult_g_scaled.mean(), np.median(mult_g_scaled)

# -------------------------------------------------------------------------
# 2. CONFIGURACIÓN DEL LIENZO HÍBRIDO (GridSpec)
# -------------------------------------------------------------------------
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.5))

# Proporciones estrictas: 3 partes para el histograma y 1 para el boxplot/violin layer
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.00)

ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# Paleta clásica de Física de Altas Energías (LHC style)
Q_COLOR = "#1f77b4"
G_COLOR = "#ff7f0e"

# -------------------------------------------------------------------------
# 3. PANEL SUPERIOR: HISTOGRAMAS DE PASO EN RANGO [-1, 1]
# -------------------------------------------------------------------------
# Generamos un set de bins denso confinado estrictamente en el rango dinámico de la KAN
bins_scaled = np.linspace(-1.0, 1.0, 35)

ax_hist.hist(
    mult_q_scaled,
    bins=bins_scaled,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=f"Quark Jets ($\mu_{{scaled}}={mean_q:.2f}$)",
)
ax_hist.hist(
    mult_g_scaled,
    bins=bins_scaled,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=f"Gluon Jets ($\mu_{{scaled}}={mean_g:.2f}$)",
)

# Líneas verticales indicadoras de la media en el espacio latente transformado
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)

# Formateo estricto del panel superior
ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    "KAN Input Space: Scaled Multiplicity Distribution [$N_{scaled} \in [-1, 1]$]",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True,
                facecolor="white",
                edgecolor="none",
                loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# Ocultar ticks del eje X compartidos para evitar solapamientos gráficos
plt.setp(ax_hist.get_xticklabels(), visible=False)

# -------------------------------------------------------------------------
# 4. PANEL INFERIOR: CAPAS HÍBRIDAS HORIZONTALES EN EL ESPACIO TRANSFORMADO
# -------------------------------------------------------------------------
# DataFrame intermedio minimalista únicamente para la API de Seaborn
df_box = pd.DataFrame(
    {
        "Scaled Multiplicity": np.concatenate([mult_q_scaled, mult_g_scaled]),
        "Class": ["Quark (Signal)"] * len(mult_q_scaled) + ["Gluon (Bkg)"] * len(mult_g_scaled),
    }
)

# Capa 1: Densidad suavizada de fondo (Violinplot)
sns.violinplot(
    x="Scaled Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    bw_method="silverman",
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Capa 2: Diagrama de cajas estrecho (Cuartiles exactos en el rango latente)
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}
flierprops = dict(marker=".", markersize=1.5, alpha=0.05, markeredgecolor="gray")

sns.boxplot(
    x="Scaled Multiplicity",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    flierprops=flierprops,
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# Formateo del panel inferior acotado al dominio estricto de la KAN
ax_box.set_xlabel("Scaled Multiplicity ($N_{scaled}$ Input)", fontsize=13)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(-1.05, 1.05)  # Muestra el rango [-1, 1] con márgenes limpios de visualización
ax_box.grid(True, linestyle=":", alpha=0.5)

# --- METADATOS ESTILO SIMULACIÓN HEP ---
fig.suptitle(
    r"Pythia 8.226 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Jet Selection Criteria: $p_T^{jet} \in [500, 550]$ GeV, $|y^{jet}| < 1.7$",
    fontsize=10,
    y=0.98,
    x=0.8,
    style="italic"
)

plt.tight_layout()
plt.show()

# -------------------------------------------------------------------------
# 5. LIBERACIÓN RIGUROSA DE MEMORIA CACHÉ EN RAM
# -------------------------------------------------------------------------
del (df_box,
     n_scaled,
     mult_q_scaled,
     mult_g_scaled,
     mean_q,
     mean_g,
     median_q,
     median_g,
     ax_hist,
     ax_box,
     fig,
     gs,
     meanprops,
     flierprops,
     bins_scaled
     )
gc.collect()

In [ ]:
# ==============================================================================
# 1. Preprocessing Vectorized with Polars
# ==============================================================================
# Extraemos los canales de pT y PDG ID de las partículas sobrevivientes del PASO 3
pt_channels = X[:, :, 0]
pdg_channels = X[:, :, 3].astype(np.int32)

# Máscaras booleanas por clase de Jet (Quark vs Gluon) donde el pT > 0 (partículas reales)
mask_q = (y == 1)[:, None] & (pt_channels > 0.0)
mask_g = (y == 0)[:, None] & (pt_channels > 0.0)

# Filtrado in-place extrayendo arreglos 1D limpios sin padding
pdgs_in_quarks = pdg_channels[mask_q]
pdgs_in_gluons = pdg_channels[mask_g]

del pt_channels, pdg_channels, mask_q, mask_g
gc.collect()

# Official Particle Data Group (PDG) dictionary for LaTeX rendering
pdg_labels = {
    211: r"$\pi^+$",
    -211: r"$\pi^-$",
    111: r"$\pi^0$",
    22: r"$\gamma$",
    321: r"$K^+$",
    -321: r"$K^-$",
    2212: r"$p$",
    -2212: r"$\bar{p}$",
    11: r"$e^-$",
    -11: r"$e^+$",
    13: r"$\mu^-$",
    -13: r"$\mu^+$",
    2112: r"$n$",
    -2112: r"$\bar{n}$",
    130: r"$K_L^0$",
    310: r"$K_S^0$",
}

# Find the top 8 most abundant particles dynamically
# (Se mantiene el límite superior analítico de 14 elementos de tu código)
all_active_pdgs = np.concatenate([pdgs_in_quarks, pdgs_in_gluons])
unique_elements, counts = np.unique(all_active_pdgs, return_counts=True)
top_particles = unique_elements[np.argsort(-counts)][:14].tolist()

del all_active_pdgs, unique_elements, counts
gc.collect()

# Filter by the top 8 and calculate the fraction normalized by Jet type (target)
total_q_particles = len(pdgs_in_quarks)
total_g_particles = len(pdgs_in_gluons)

# Extract the final vectors ready for Matplotlib
frac_g = np.array([np.sum(pdgs_in_gluons == p) / total_g_particles for p in top_particles], dtype=np.float32)
frac_q = np.array([np.sum(pdgs_in_quarks == p) / total_q_particles for p in top_particles], dtype=np.float32)

del pdgs_in_quarks, pdgs_in_gluons
gc.collect()

# ==============================================================================
# 2. RENDERING THE DISCRETE BAR PLOT
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(8.0, 5.5))

x_indexes = np.arange(len(top_particles))
bar_width = 0.35

# Draw the structured bars
ax.bar(
    x_indexes - bar_width / 2,
    frac_g,
    bar_width,
    color=G_COLOR,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
    label="Gluon Jets (Background)",
)
ax.bar(
    x_indexes + bar_width / 2,
    frac_q,
    bar_width,
    color=Q_COLOR,
    alpha=0.85,
    edgecolor="black",
    linewidth=0.8,
    label="Quark Jets (Signal)",
)

# Map the names in LaTeX format on the X axis
xtick_names = [pdg_labels.get(int(p), f"ID: {int(p)}") for p in top_particles]
ax.set_xticks(x_indexes)
ax.set_xticklabels(xtick_names, rotation=0, fontsize=12)

ax.set_xlabel("Particle Identity (PDG Code)", fontsize=13)
ax.set_ylabel("Relative Fraction per Jet Type", fontsize=13)
ax.set_title(
    "Jet Particle Composition (Hadronization Layer)",
    loc="left",
    fontweight="bold",
    fontsize=14,
)

fig.suptitle(
    r"Pythia 8.226 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Jet Selection Criteria: $p_T^{jet} \in [500, 550]$ GeV, $|y^{jet}| < 1.7$",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

# Style formatting details
ax.tick_params(direction="in", top=False, right=True, labelsize=11)
ax.grid(True, axis="y", linestyle=":", alpha=0.6)
ax.legend(frameon=True, facecolor="white", edgecolor="none", fontsize=11)

plt.tight_layout()
plt.show()

# Strict memory cleanup
del (
    pdg_labels,
    top_particles,
    frac_g,
    frac_q,
    fig,
    ax,
    x_indexes,
    bar_width,
    xtick_names,
    total_q_particles,
    total_g_particles
)
gc.collect()

In [ ]:
# ==============================================================================
# 1. Electric Charge Extraction and Vectorized Preprocessing
# ==============================================================================
# Extracción directa de la columna 1 correspondiente al espacio transformado de Q_jet
q_jet_scaled = processed_matrix[:, 1]

# Class Separation (0: Gluon, 1: Quark) mediante máscaras booleanas directas
q_jet_q = q_jet_scaled[y == 1]
q_jet_g = q_jet_scaled[y == 0]

# Statistical metrics for the classes
mean_q, median_q = q_jet_q.mean(), np.median(q_jet_q)
mean_g, median_g = q_jet_g.mean(), np.median(q_jet_g)

# ==============================================================================
# 3. HYBRID CANVAS LAYOUT (GridSpec Vertical)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.06)
ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# ==============================================================================
# 4.  UPPER PANEL: ADJUSTED DENSITIES (HISTOGRAMS)
# ==============================================================================
bins_qjet = np.linspace(-1.0, 1.0, 50)

ax_hist.hist(
    q_jet_q,
    bins=bins_qjet,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=rf"Quark Jets ($\mu={mean_q:.3f}$)",
)
ax_hist.hist(
    q_jet_g,
    bins=bins_qjet,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=rf"Gluon Jets ($\mu={mean_g:.3f}$)",
)

# Vertical lines indicating the means of the distributions
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(
    0.0, color="gray", linestyle=":", lw=1.0
)  # Reference for pure neutrality

ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    "Jet Charge Distribution: Quantum Flavor Asymmetry",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 5.  LOWER PANEL: VIOLINPLOT + BOXPLOT COMBINED
# ==============================================================================
df_box = pd.DataFrame(
    {
        "JetCharge": np.concatenate([q_jet_q, q_jet_g]),
        "Class": ["Quark (Signal)"] * len(q_jet_q) + ["Gluon (Bkg)"] * len(q_jet_g),
    }
)

# Smoothed background violinplot
sns.violinplot(
    x="JetCharge",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    bw_method="silverman",
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Narrow boxplot for exact quartiles
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}
flierprops = dict(marker=".", markersize=1.5, alpha=0.02, markeredgecolor="gray")

sns.boxplot(
    x="JetCharge",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    flierprops=flierprops,
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# Fixed annotation of absolute minimum and maximum at the inner extremes of the panel
ax_box.text(
    -0.75,
    -0.2,
    f"Quark Range: [{q_jet_q.min():.2f}, {q_jet_q.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=Q_COLOR,
    ha="left",
)
ax_box.text(
    -0.75,
    1.25,
    f"Gluon Range: [{q_jet_g.min():.2f}, {q_jet_g.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=G_COLOR,
    ha="left",
)

# Lower panel formatting
ax_box.set_xlabel(
    r"Weighted Jet Charge ($Q_{\mathrm{jet}}$ with $\kappa=0.5$)", fontsize=13
)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
ax_box.set_xlim(-1.0, 1.0)  # Optimal symmetric centering for U(1) physics
ax_box.grid(True, linestyle=":", alpha=0.5)

# Hide X-axis labels of the upper histogram to avoid typographic overlap
plt.setp(ax_hist.get_xticklabels(), visible=False)

# Collider metadata (HEP Style)
fig.suptitle(
    r"Pythia 8.2 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Kinematic Window: $p_T^{jet} \in [500, 550]$ GeV, Pondered by Local $p_T$ Fractions",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

print("Mínimo de la carga graficada:", q_jet_scaled.min())
print("Máximo de la carga graficada:", q_jet_scaled.max())

# ==============================================================================
# 6. RIGOROUS RAM CLEANUP
# ==============================================================================
del (
    q_jet_scaled,
    df_box,
    mean_q,
    mean_g,
    median_q,
    median_g
)
del (
    fig,
    gs,
    ax_hist,
    ax_box,
    bins_qjet,
    meanprops,
    flierprops
)
gc.collect()

In [ ]:
# ==============================================================================
# 1. EXTRACCIÓN Y CÁLCULO VECTORIZADO DE DELTA R
# ==============================================================================
# Segregación por clases usando las etiquetas reales (0: Gluón, 1: Quark)
dr_q = processed_matrix[y == 1][:, 3::2].flatten()  # Delta R para Quarks
dr_g = processed_matrix[y == 0][:, 3::2].flatten()  # Delta R para Gluones

# Métricas estadísticas descriptivas para las marcas e inyección de texto
mean_q, median_q = dr_q.mean(), np.median(dr_q)
mean_g, median_g = dr_g.mean(), np.median(dr_g)

# ==============================================================================
# 2. CONFIGURACIÓN DEL CANVAS HÍBRIDO (GridSpec Vertical)
# ==============================================================================
plt.style.use("seaborn-v0_8-whitegrid")
fig = plt.figure(figsize=(9.0, 7.0))

# División 75/25 para el histograma superior y los diagramas de caja inferiores
gs = GridSpec(2, 1, height_ratios=[3, 1], hspace=0.06)
ax_hist = fig.add_subplot(gs[0])
ax_box = fig.add_subplot(gs[1], sharex=ax_hist)

# Colores consistentes con tu pipeline de subestructura
#Q_COLOR = "#1f77b4"  # Azul para Quark (Signal)
#G_COLOR = "#e377c2"  # Rosa/Menta para Gluón (Background)

# ==============================================================================
# 3. PANEL SUPERIOR: DENSIDADES DE PASOS (HISTOGRAMAS 1D)
# ==============================================================================
# Límites acotados físicamente por el radio del algoritmo anti-kt (R=0.4) + margen
bins_dr = np.linspace(0.0, 1.0, 110)

ax_hist.hist(
    dr_q,
    bins=bins_dr,
    histtype="step",
    lw=2.2,
    color=Q_COLOR,
    density=True,
    label=rf"Quark Jets ($\mu={mean_q:.3f}$)",
)
ax_hist.hist(
    dr_g,
    bins=bins_dr,
    histtype="step",
    lw=2.2,
    color=G_COLOR,
    density=True,
    label=rf"Gluon Jets ($\mu={mean_g:.3f}$)",
)

# Líneas discontinuas indicadoras de los valores medios
ax_hist.axvline(mean_q, color=Q_COLOR, linestyle="--", lw=1.5, alpha=0.8)
ax_hist.axvline(mean_g, color=G_COLOR, linestyle="--", lw=1.5, alpha=0.8)

ax_hist.set_ylabel("Normalized Density", fontsize=13)
ax_hist.set_title(
    r"Jet Substructure: Radial Constituent Dispersion ($\Delta R$)",
    loc="left",
    fontweight="bold",
    fontsize=14,
)
ax_hist.legend(frameon=True, facecolor="white", edgecolor="none", loc="upper right")
ax_hist.grid(True, linestyle=":", alpha=0.5)

# ==============================================================================
# 4. PANEL INFERIOR: VIOLINPLOT + BOXPLOT HORIZONTAL CONTRAPUESTO
# ==============================================================================
df_box = pd.DataFrame(
    {
        "DeltaR": np.concatenate([dr_q, dr_g]),
        "Class": ["Quark (Signal)"] * len(dr_q) + ["Gluon (Bkg)"] * len(dr_g),
    }
)

# Capa inferior: Violinplot de densidad suavizada
sns.violinplot(
    x="DeltaR",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    inner=None,
    linewidth=0.8,
    edgecolor="gray",
    alpha=0.25,
    ax=ax_box,
)

# Capa superior: Boxplot angosto optimizado (showfliers=False para evitar PDF pesado)
meanprops = {
    "marker": "o",
    "markerfacecolor": "white",
    "markeredgecolor": "black",
    "markersize": 4.5,
}

sns.boxplot(
    x="DeltaR",
    y="Class",
    data=df_box,
    hue="Class",
    legend=False,
    palette=[Q_COLOR, G_COLOR],
    orient="h",
    showmeans=True,
    meanprops=meanprops,
    showfliers=False,  # Evita que miles de puntos vectoriales inflen el peso del archivo
    width=0.22,
    linewidth=1.2,
    ax=ax_box,
)

# Anotación fija de mínimos y máximos en el extremo superior interno
ax_box.text(
    0.02,
    -0.2,
    f"Quark Range: [{dr_q.min():.2f}, {dr_q.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=Q_COLOR,
    ha="left",
)
ax_box.text(
    0.02,
    0.85,
    f"Gluon Range: [{dr_g.min():.2f}, {dr_g.max():.2f}]",
    fontsize=9,
    fontweight="bold",
    color=G_COLOR,
    ha="left",
)

# Formato estricto del eje compartido
ax_box.set_xlabel(
    r"Constituent Angular Distance to Jet Axis ($\Delta R = \sqrt{\Delta\eta^2 + \Delta\phi^2}$)",
    fontsize=13,
)
ax_box.set_ylabel("")
ax_box.set_yticklabels(["Quark\n(Signal)", "Gluon\n(Bkg)"], fontsize=11)
#ax_box.set_xlim(-0.02, 1.0)  # Acotado al tamaño del cono real
ax_box.grid(True, linestyle=":", alpha=0.5)

# Ocultar etiquetas repetitivas del eje X superior
plt.setp(ax_hist.get_xticklabels(), visible=False)

# Metadatos del experimento (Estilo Publicación HEP)
fig.suptitle(
    r"Pythia 8.2 Simulation, $\sqrt{s} = 14$ TeV, anti-$k_t$ ($R=0.4$)"
    "\n"
    r"Kinematic Selection: $p_T^{jet} \in [500, 550]$ GeV, All Resolved Constituents",
    fontsize=10,
    y=0.97,
    style="italic",
    ha="center",
)

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. LIMPIEZA TOTAL DE MEMORIA RAM
# ==============================================================================
del dr_q, dr_g, df_box
del fig, gs, ax_hist, ax_box, bins_dr, meanprops
gc.collect()

In [ ]:
# ==============================================================================
# 1. Preprocessing Vectorized for 2D Angular Distributions
# ==============================================================================
import gc
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Supongamos que cuentas con:
# - X: np.ndarray de forma (N, N_cut, 4) tras el truncamiento por Pareto
# - y: np.ndarray de forma (N,) -> 0: Gluon, 1: Quark

pt_raw = X[:, :, 0]
eta_raw = X[:, :, 1]
phi_raw = X[:, :, 2]

sum_pt = np.sum(pt_raw, axis=1)
sum_pt_safe = np.where(sum_pt == 0, 1.0, sum_pt)

# Cálculo exacto del baricentro del Jet
eta_jet = np.sum(pt_raw * eta_raw, axis=1) / sum_pt_safe

sin_phi_avg = np.sum(pt_raw * np.sin(phi_raw), axis=1) / sum_pt_safe
cos_phi_avg = np.sum(pt_raw * np.cos(phi_raw), axis=1) / sum_pt_safe
phi_jet = np.arctan2(sin_phi_avg, cos_phi_avg)

# Cálculo vectorizado de las diferencias respecto al eje del jet
d_eta = eta_raw - eta_jet[:, None]
d_phi = np.arctan2(np.sin(phi_raw - phi_jet[:, None]), np.cos(phi_raw - phi_jet[:, None]))
d_R = np.sqrt(d_eta**2 + d_phi**2)

# --- APLICACIÓN ESTRICTA DE LAS MÁSCARAS DEL PIPELINE ---
# Una partícula es válida si pasa el filtro infrarrojo Y está dentro del cono R <= 0.4
valid_physics_mask = (pt_raw > 1e-3) & (d_R <= 0.4)

# Máscaras lógicas finales combinando la clase de Jet con la validez física de la partícula
mask_q = (y == 1)[:, None] & valid_physics_mask
mask_g = (y == 0)[:, None] & valid_physics_mask

# Aplanar los arreglos conservando únicamente los constituyentes purificados que van a la KAN
d_eta_q = d_eta[mask_q]/0.4
d_phi_q = d_phi[mask_q]/0.4

d_eta_g = d_eta[mask_g]/0.4
d_phi_g = d_phi[mask_g]/0.4

del pt_raw, eta_raw, phi_raw, sum_pt, sum_pt_safe, eta_jet, sin_phi_avg, cos_phi_avg, phi_jet, d_eta, d_phi, d_R, valid_physics_mask, mask_q, mask_g
gc.collect()

# ==============================================================================
# 2. RENDERING THE DISCRETE BAR PLOT / 2D HISTOGRAMS
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6.5), sharex=True, sharey=True)

# Adjust the range to the actual jet size (R = 0.4) plus a small margin
img_range = [[-1.5, 1.5], [-1.5, 1.5]]
bin_resolution = 120

# --- Panel 1: Quarks (Signal) ---
im0 = axes[0].hist2d(
    d_eta_q,
    d_phi_q,
    bins=bin_resolution,
    range=img_range,
    cmap="inferno",
    norm=LogNorm(),
)
cbar0 = fig.colorbar(im0[3], ax=axes[0], fraction=0.046, pad=0.04)
cbar0.set_label("Constituent Density (Log Scale)", rotation=270, labelpad=15)
axes[0].set_title(
    "Signal: Quark Jets", fontsize=14, fontweight="bold"
)
axes[0].set_xlabel(r"$\Delta \eta = \eta_{part} - \eta_{jet}$", fontsize=14)
axes[0].set_ylabel(r"$\Delta \phi = \phi_{part} - \phi_{jet}$", fontsize=14)

# --- Panel 2: Gluons (Background) ---
im1 = axes[1].hist2d(
    d_eta_g,
    d_phi_g,
    bins=bin_resolution,
    range=img_range,
    cmap="inferno",
    norm=LogNorm(),
)
cbar1 = fig.colorbar(im1[3], ax=axes[1], fraction=0.046, pad=0.04)
cbar1.set_label("Constituent Density (Log Scale)", rotation=270, labelpad=15)
axes[1].set_title(
    "Background: Gluon Jets", fontsize=14, fontweight="bold"
)
axes[1].set_xlabel(r"$\Delta \eta = \eta_{part} - \eta_{jet}$", fontsize=14)

# Strict formatting for both panels
for ax in axes:
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(img_range[0])
    ax.set_ylim(img_range[1])
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(True, linestyle="--", alpha=0.4)

plt.suptitle(
    "Jet Substructure: Relative Phase Space Geometry ($q/g$ Separation)",
    fontsize=18,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

# Strict memory cleanup
del d_eta_q, d_phi_q, d_eta_g, d_phi_g, img_range, bin_resolution, fig, axes
gc.collect()